# Day 32 — Mini Project #1: Full EDA on Telco Customer ChurnDataset: [Telco Customer Churn (Kaggle)](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)Goal: Full pipeline — load, clean, explore, apply stats, and summarize insights on customer churn.

## 1. Kickoff — Setup & Load

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom scipy.stats import skew, ttest_ind

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')df.shape

In [ ]:
df.head()

## 2. Scouting Report — Initial Inspection

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.dtypes

**Observation:** `TotalCharges` is showing as `object` even though it's a monetary value — should be `float64`. This gets fixed in the next section.

## 3. Fitness Check — Data Cleaning

In [ ]:
# Fix TotalCharges dtype — invalid entries (blank strings) become NaNdf['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')df.isnull().sum()

**11 missing values in `TotalCharges`** — these are new customers with 0 tenure who haven't been billed yet. Filling with 0 rather than dropping, since a customer with 0 tenure logically has 0 total charges, and dropping rows would lose their other attribute data unnecessarily.

In [ ]:
df['TotalCharges'] = df['TotalCharges'].fillna(0)df['TotalCharges'].isnull().sum()

In [ ]:
# Check for duplicatesprint("Full-row duplicates:", df.duplicated().sum())print("Duplicate customerIDs:", df['customerID'].duplicated().sum())

No duplicates found — every customer record is unique.

## 4. Formation Analysis — Univariate EDA

In [ ]:
df['Churn'].value_counts()

In [ ]:
df['Churn'].value_counts(normalize=True) * 100

In [ ]:
sns.countplot(x='Churn', data=df)plt.title('Churn Distribution')plt.show()

**Churn is imbalanced** — roughly 73% No vs 27% Yes. Most customers stay; a minority churn.

In [ ]:
sns.histplot(df['tenure'], kde=True)plt.title('Tenure Distribution')plt.show()

In [ ]:
sns.histplot(df['MonthlyCharges'], kde=True)plt.title('Monthly Charges Distribution')plt.show()

In [ ]:
sns.histplot(df['TotalCharges'], kde=True)plt.title('Total Charges Distribution')plt.show()

In [ ]:
print("Tenure skew:", skew(df['tenure']))print("MonthlyCharges skew:", skew(df['MonthlyCharges']))print("TotalCharges skew:", skew(df['TotalCharges']))

- Tenure: roughly balanced (slight right skew)- MonthlyCharges: slightly left-skewed- TotalCharges: right-skewed — most customers have lower total charges, with a long tail of long-tenure customers who've paid much more over time

## 5. Head-to-Head — Bivariate / Groupby EDA

In [ ]:
df.groupby('Contract')['Churn'].value_counts(normalize=True)

**Month-to-month contracts churn the most.** No long-term commitment makes it easier to leave whenever. One-year and two-year contracts lock customers in, so churn is much lower there.

In [ ]:
df.groupby('InternetService')['Churn'].value_counts(normalize=True)

In [ ]:
df.groupby('PaymentMethod')['Churn'].value_counts(normalize=True)

**Electronic check has the highest churn rate among payment methods.** It's the one method without auto-pay, so there's more friction to stay — customers have to actively remember to pay each cycle, making it easier to walk away.

In [ ]:
sns.boxplot(x='Churn', y='tenure', data=df)plt.title('Tenure by Churn Status')plt.show()

In [ ]:
df.groupby('Churn')['tenure'].median()

**Churned customers stay for much less time** (median 10 months) compared to loyal customers (median 38 months) — nearly a 4x difference.

## 6. VAR Review — Stats Application

In [ ]:
numeric_cols = df[['tenure', 'MonthlyCharges', 'TotalCharges']]sns.heatmap(numeric_cols.corr(), annot=True, cmap='coolwarm')plt.title('Correlation Heatmap')plt.show()

**Hypothesis test:** Is there a significant difference in tenure between churned and non-churned customers?- H0: No difference in mean tenure between the two groups- H1: There is a difference in mean tenure between the two groups- Test: Two-sample independent t-test (comparing two separate, independent groups against each other)

In [ ]:
churned = df[df['Churn'] == 'Yes']['tenure']not_churned = df[df['Churn'] == 'No']['tenure']t_stat, p_value = ttest_ind(churned, not_churned)print("t-statistic:", t_stat)print("p-value:", p_value)

**Conclusion:** With a p-value of ~8e-205 (far below 0.05), there is strong statistical evidence that churned customers have significantly shorter average tenure than non-churned customers.

## 7. Full-Time Whistle — Insights Summary- **Churn is imbalanced** — roughly 73% No vs 27% Yes- **Contract type matters** — month-to-month customers churn the most since there's no long-term commitment; one/two-year contracts churn far less- **Payment method matters** — electronic check payers churn the most, likely due to lack of auto-pay creating more friction to stay- **Tenure is the strongest signal** — churned customers leave much earlier (median 10 months) than loyal customers (median 38 months), a gap confirmed with a two-sample t-test (p ≈ 8e-205)